# Advanced Modeling Features

This notebook demonstrates advanced modeling capabilities in ws3:

- **Stochastic Optimization**: Handle uncertainty in growth, prices, and disturbances
- **Multi-Objective Optimization**: Trade-off analysis between competing objectives
- **Dynamic Planning**: Re-optimization over multiple time periods
- **Climate Scenarios**: Integration of climate change projections
- **Enhanced Carbon Accounting**: Detailed carbon pool modeling

**Prerequisites:** Completion of `070_ws3_quickstart_complete_workflow.ipynb`

**Note:** This notebook requires the `ws3.advanced_modeling` module.

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ws3.forest
import ws3.opt
from ws3.advanced_modeling import (
    StochasticOptimizer,
    MultiObjectiveOptimizer,
    DynamicPlanner,
    ClimateScenarioManager,
    UncertaintyType,
    create_stochastic_optimizer,
    create_multi_objective_optimizer,
    create_dynamic_planner,
    create_climate_manager
)

print("Imports complete!")

## 1. Stochastic Optimization

Forest management involves significant uncertainty:

- **Growth uncertainty**: Weather, pests, diseases affect tree growth
- **Price uncertainty**: Market fluctuations for timber products
- **Demand uncertainty**: Changes in wood product demand
- **Disturbance uncertainty**: Fire, insects, windthrow events
- **Climate uncertainty**: Long-term climate change impacts

Stochastic optimization handles this uncertainty by considering multiple scenarios and optimizing expected outcomes.

**Approaches:**
- **Sample Average Approximation (SAA)**: Solve each scenario and average results
- **Scenario Reduction**: Reduce number of scenarios while preserving key characteristics
- **Robust Optimization**: Optimize for worst-case scenario

In [ ]:
# Create a simple optimization problem for demonstration
# (In practice, you would use a real ForestModel)

from ws3.opt import Problem

# Create problem with growth uncertainty
problem = Problem()

# Add variables for harvest in each period
for t in range(5):
    problem.add_variable(f"harvest_t{t}", "continuous", 0, 1000)

# Add even-flow constraint
problem.add_constraint(
    "even_flow",
    {f"harvest_t{t}": 1.0 for t in range(5)},
    "=",
    2000  # Total harvest
)

# Set objective (simplified NPV)
discount_rate = 0.05
objective = {}
for t in range(5):
    npv_factor = 1.0 / (1 + discount_rate) ** t
    objective[f"harvest_t{t}"] = npv_factor * 100

problem.set_objective(objective)

print("Problem created:")
print(f"  Variables: {len(problem._vars)}")
print(f"  Constraints: {len(problem._constraints)}")

# Create stochastic optimizer
stoch_optimizer = create_stochastic_optimizer(problem)

# Generate growth uncertainty scenarios
print("Generating growth uncertainty scenarios...")
scenarios = stoch_optimizer.generate_scenarios(
    uncertainty_type=UncertaintyType.GROWTH,
    n_scenarios=20,
    mean=1.0,
    std=0.2
)

print(f"Generated {len(scenarios)} scenarios")

# Show scenario summary
summary = stoch_optimizer.get_scenario_summary()
print("\nScenario Summary:")
print(summary.head(10))

# Solve stochastic problem using sample average
print("\nSolving stochastic problem (Sample Average Approximation)...")
results = stoch_optimizer.solve_stochastic(method="sample_average")

print(f"\nResults:")
print(f"  Expected value: {results['expected_value']:.2f}")
print(f"  Standard deviation: {results['std_dev']:.2f}")
print(f"  Number of scenarios: {results['n_scenarios']}")

In [ ]:
## 2. Multi-Objective Optimization

Forest management often involves multiple, potentially conflicting objectives:

- **Economic**: Maximize net present value (NPV)
- **Ecological**: Maximize carbon sequestration
- **Social**: Maintain even-flow harvest levels
- **Conservation**: Protect habitat quality

Multi-objective optimization finds trade-off solutions that balance these competing goals.

**Methods:**
- **Weighted Sum**: Combine objectives into single objective with weights
- **Epsilon-Constraint**: Optimize one objective while constraining others
- **Pareto Frontier**: Find all non-dominated solutions

# Create multi-objective optimizer
mo_optimizer = create_multi_objective_optimizer(problem)

# Add multiple objectives
mo_optimizer.add_objective("npv", weight=1.0, direction="maximize")
mo_optimizer.add_objective("even_flow", weight=0.5, direction="minimize_deviation")
mo_optimizer.add_objective("carbon", weight=0.3, direction="maximize")

print(f"Added {len(mo_optimizer.objectives)} objectives")

# Solve with different weight combinations
print("\nSolving with different weight combinations...")
weight_sets = [
    {"npv": 0.8, "even_flow": 0.1, "carbon": 0.1},
    {"npv": 0.5, "even_flow": 0.3, "carbon": 0.2},
    {"npv": 0.2, "even_flow": 0.3, "carbon": 0.5},
]

results = []
for weights in weight_sets:
    result = mo_optimizer.solve_weighted_sum(weights)
    results.append({
        'weights': weights,
        'objective_values': result['objective_values']
    })
    print(f"Weights: {weights}")
    print(f"  Objectives: {result['objective_values']}")
    print()

# Find Pareto frontier
print("Finding Pareto frontier...")
pareto_frontier = mo_optimizer.find_pareto_frontier(n_points=10)
print(f"Found {len(pareto_frontier)} Pareto-optimal solutions")
print(pareto_frontier)

In [ ]:
## 3. Dynamic Planning

Dynamic planning involves re-optimizing harvest schedules over multiple time periods:

- **Initial plan**: Optimize for full planning horizon
- **Re-optimization**: Update plan based on new information
- **Adaptive management**: Adjust to changing conditions

**Benefits:**
- Incorporates new information (growth, prices, disturbances)
- Reduces risk from uncertainty
- Allows for adaptive management strategies

**Approaches:**
- **Rolling horizon**: Re-optimize every N periods
- **Scenario-based**: Consider multiple future scenarios
- **Robust**: Optimize for worst-case scenarios

# Create dynamic planner
dyn_planner = create_dynamic_planner(problem, n_periods=10)

# Generate static plan
print("Generating static plan...")
static_plan = dyn_planner.plan_static()
print(f"Static plan objective: {static_plan['objective_value']:.2f}")

# Generate dynamic plan (re-optimize every 5 periods)
print("\nGenerating dynamic plan (re-optimize every 5 periods)...")
dynamic_plan = dyn_planner.plan_dynamic(reoptimize_every=5)
print(f"Dynamic plan total objective: {dynamic_plan['total_objective']:.2f}")

# Compare plans
comparison = dyn_planner.compare_plans(static_plan, dynamic_plan)
print("\nPlan Comparison:")
print(f"  Static objective: {comparison['plan1_objective']:.2f}")
print(f"  Dynamic objective: {comparison['plan2_objective']:.2f}")
print(f"  Improvement: {comparison['improvement']:.2f}%")

In [ ]:
## 4. Climate Scenarios

Climate change affects forest growth, disturbance regimes, and carbon dynamics:

- **Temperature increase**: Affects growth rates and species distributions
- **Precipitation changes**: Affects water availability and growth
- **CO2 fertilization**: May enhance growth
- **Disturbance frequency**: Fire, insects, windthrow may increase

**Integration with Optimization:**
- Modify yield curves based on climate scenarios
- Adjust disturbance probabilities
- Update carbon accounting models

**Applications:**
- Climate-adaptive harvest planning
- Carbon offset projects
- Conservation planning under climate change

# Create climate scenario manager
climate_manager = create_climate_manager()

# Add RCP scenarios
rcp_scenarios = climate_manager.get_rcp_scenarios()
print("RCP Scenarios:")
for scenario in rcp_scenarios:
    print(f"  {scenario['name']}: +{scenario['temperature']}°C, "
          f"{scenario['precipitation']*100:+.0f}% precip, "
          f"{scenario['co2']} ppm CO2")

# In practice, you would:
# 1. Load a real ForestModel
# 2. Run optimization under each climate scenario
# 3. Compare results

print("\nClimate analysis would:")
print("  1. Modify yield curves based on temperature/precipitation")
print("  2. Adjust disturbance probabilities")
print("  3. Run optimization for each scenario")
print("  4. Compare harvest schedules and objectives")
print("  5. Identify climate-adaptive management strategies")

In [ ]:
## Summary

**Advanced Modeling Features:**

| Feature | Purpose | Key Methods |
|---------|---------|-------------|
| Stochastic Optimization | Handle uncertainty | SAA, scenario reduction, robust optimization |
| Multi-Objective | Balance competing goals | Weighted sum, epsilon-constraint, Pareto frontier |
| Dynamic Planning | Adaptive management | Rolling horizon, re-optimization |
| Climate Scenarios | Climate-adaptive planning | RCP scenarios, growth modification |

**Key Takeaways:**

1. **Stochastic Optimization**: Incorporates uncertainty through scenario-based approaches
2. **Multi-Objective**: Finds trade-off solutions between competing objectives
3. **Dynamic Planning**: Allows adaptive management with periodic re-optimization
4. **Climate Scenarios**: Integrates climate projections into harvest planning

**When to Use Each:**

- Use **stochastic optimization** when uncertainty is significant
- Use **multi-objective** when balancing economic, ecological, and social goals
- Use **dynamic planning** for long-term adaptive management
- Use **climate scenarios** for climate-resilient planning

**Next Steps:**

- Install additional packages for full functionality
- Test with real forest inventory data
- Customize scenario generation for specific applications
- Integrate with FEMIC for detailed carbon accounting